In [2]:
# ==============================================================================
# PIPELINE MULTIAGENTE AGNO - SUPERSTORE USA (DATASET LOCAL + RESILIÊNCIA)
# Versão atualizada: modelos Gemini 3.x, fail-fast em 404 e fallback por etapa
# ==============================================================================

import os
import time
import pandas as pd
from IPython.display import Markdown, display

# 1. INSTALAÇÃO DE DEPENDÊNCIAS (Colab)
print("1/5 Instalando bibliotecas necessárias...")
!pip install -q agno google-genai tabulate "google-auth==2.49.0"

# 2. CONFIGURAÇÃO DE AMBIENTE E AUTENTICAÇÃO
print("\n2/5 Configurando chaves de acesso e diretórios...")
os.makedirs("outputs", exist_ok=True)

try:
    from google.colab import userdata
    api_key = userdata.get('GEMINI_API_KEY')
    if not api_key:
        api_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    import getpass
    api_key = getpass.getpass("Insira sua Google API Key: ")

os.environ["GOOGLE_API_KEY"] = api_key
os.environ["GEMINI_API_KEY"] = api_key
print("✓ Chaves de autenticação configuradas.")

# (Opcional) Descomente para ver exatamente quais modelos a sua chave enxerga:
# from google import genai
# for m in genai.Client(api_key=api_key).models.list():
#     print(m.name)

# 3. LEITURA DO ARQUIVO LOCAL JÁ CARREGADO
print("\n3/5 Carregando o arquivo de dados...")
caminhos_possiveis = [
    "SampleSuperstore.csv",
    "Sample - Superstore.csv",
    "data/SampleSuperstore.csv",
    "data/Sample - Superstore.csv",
]

csv_path = next((c for c in caminhos_possiveis if os.path.exists(c)), None)
if not csv_path:
    raise FileNotFoundError(
        "Arquivo 'SampleSuperstore.csv' não foi localizado. "
        "Certifique-se de que fez o upload no painel lateral de arquivos do Colab."
    )

print(f"✓ Arquivo localizado: {csv_path}")

try:
    df_superstore = pd.read_csv(csv_path, encoding='utf-8')
except UnicodeDecodeError:
    df_superstore = pd.read_csv(csv_path, encoding='latin1')

# Engenharia de atributos e conversões temporais
if 'Order Date' in df_superstore.columns:
    df_superstore['Order Date'] = pd.to_datetime(df_superstore['Order Date'])
if 'Ship Date' in df_superstore.columns:
    df_superstore['Ship Date'] = pd.to_datetime(df_superstore['Ship Date'])
    if 'Order Date' in df_superstore.columns:
        df_superstore['Shipping Days'] = (
            df_superstore['Ship Date'] - df_superstore['Order Date']
        ).dt.days

if 'Sales' in df_superstore.columns and 'Profit' in df_superstore.columns:
    df_superstore['Profit Margin %'] = (df_superstore['Profit'] / df_superstore['Sales']) * 100

print(f"✓ Base carregada com sucesso: {df_superstore.shape[0]} linhas e {df_superstore.shape[1]} colunas.")

# 4. FERRAMENTAS ANALÍTICAS DETERMINÍSTICAS (TOOLS)
print("\n4/5 Preparando ferramentas analíticas para os agentes...")

def get_executive_kpis() -> str:
    """Calcula os KPIs macro-executivos globais da Superstore."""
    total_sales = float(df_superstore['Sales'].sum())
    total_profit = float(df_superstore['Profit'].sum())
    margin = float((total_profit / total_sales) * 100) if total_sales > 0 else 0.0
    avg_discount = float(df_superstore['Discount'].mean() * 100) if 'Discount' in df_superstore.columns else 0.0

    unique_orders = int(df_superstore['Order ID'].nunique()) if 'Order ID' in df_superstore.columns else len(df_superstore)
    unique_customers = int(df_superstore['Customer ID'].nunique()) if 'Customer ID' in df_superstore.columns else 0

    kpis = {
        "Faturamento Total (USD)": f"${total_sales:,.2f}",
        "Lucro Total (USD)": f"${total_profit:,.2f}",
        "Margem Líquida Geral (%)": f"{margin:.2f}%",
        "Desconto Médio Concedido (%)": f"{avg_discount:.2f}%",
        "Volume de Pedidos Únicos": f"{unique_orders:,}",
        "Base de Clientes Ativos": f"{unique_customers:,}",
    }
    summary = "### KPIs Executivos Globais\n"
    for k, v in kpis.items():
        summary += f"- **{k}**: {v}\n"
    return summary


def get_performance_by_dimension(dimension: str = "Region") -> str:
    """Agrupa métricas de vendas e lucro por 'Region', 'Segment' ou 'Category'."""
    if dimension not in df_superstore.columns:
        return f"Coluna '{dimension}' não encontrada na base de dados."

    agg_dict = {
        'Total_Sales': ('Sales', 'sum'),
        'Total_Profit': ('Profit', 'sum'),
    }
    if 'Order ID' in df_superstore.columns:
        agg_dict['Order_Count'] = ('Order ID', 'nunique')
    else:
        agg_dict['Order_Count'] = ('Sales', 'count')

    grouped = df_superstore.groupby(dimension).agg(**agg_dict).reset_index()
    grouped['Profit_Margin_%'] = (grouped['Total_Profit'] / grouped['Total_Sales']) * 100
    grouped['Sales_Share_%'] = (grouped['Total_Sales'] / grouped['Total_Sales'].sum()) * 100
    grouped = grouped.sort_values(by='Total_Sales', ascending=False)
    return f"### Análise de Performance por {dimension}\n" + grouped.to_markdown(index=False)


def get_product_and_shipping_diagnostics() -> str:
    """Identifica subcategorias deficitárias e avalia logística por modal de envio."""
    resultado = ""
    if 'Sub-Category' in df_superstore.columns:
        subcats = df_superstore.groupby('Sub-Category').agg(
            Total_Sales=('Sales', 'sum'),
            Total_Profit=('Profit', 'sum'),
            Avg_Discount=('Discount', 'mean') if 'Discount' in df_superstore.columns else ('Sales', 'count'),
        ).reset_index()
        subcats['Margin_%'] = (subcats['Total_Profit'] / subcats['Total_Sales']) * 100
        deficitarias = subcats.sort_values(by='Total_Profit', ascending=True).head(5)
        resultado += "### Top 5 Subcategorias com Maior Prejuízo Financeiro:\n" + deficitarias.to_markdown(index=False) + "\n\n"

    if 'Ship Mode' in df_superstore.columns:
        ship_agg = {
            'Total_Sales': ('Sales', 'sum'),
            'Total_Orders': ('Order ID', 'nunique') if 'Order ID' in df_superstore.columns else ('Sales', 'count'),
        }
        if 'Shipping Days' in df_superstore.columns:
            ship_agg['Avg_Delivery_Days'] = ('Shipping Days', 'mean')

        shipping = (
            df_superstore.groupby('Ship Mode').agg(**ship_agg).reset_index()
            .sort_values(by='Total_Sales', ascending=False)
        )
        resultado += "### Eficiência Operacional por Modal de Frete:\n" + shipping.to_markdown(index=False)

    return resultado if resultado else "Dados de produtos ou logística não identificados."


# 5. AGENTES, MODELOS E RESILIÊNCIA
print("\n5/5 Configurando e executando os agentes...")
from agno.agent import Agent
from agno.models.google import Gemini

# Ordem: do mais barato ao mais capaz. IDs verificados na documentação do Gemini API.
# Obs.: gemini-2.5-* está em aposentadoria (16/10/2026) e foi removido da lista.
MODEL_IDS = [
    "gemini-3.5-flash-lite",
    "gemini-3.5-flash",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
]

# Definição dos 4 agentes (instruções separadas da escolha do modelo)
AGENT_SPECS = {
    "analyst": dict(
        name="Senior Data Analyst",
        tools=[get_executive_kpis, get_performance_by_dimension, get_product_and_shipping_diagnostics],
        instructions=[
            "Você é o analista quantitativo sênior.",
            "Use as ferramentas disponíveis para obter os números exatos.",
            "Retorne todos os dados consolidados e tabulados sem arredondar incorretamente ou omitir informações.",
        ],
    ),
    "ceo": dict(
        name="CEO Strategic Reporter",
        tools=[],
        instructions=[
            "Você é o Chief of Staff responsável pelo relatório estratégico para o CEO e Acionistas.",
            "Mantenha um tom conciso, executivo e focado em margens, riscos e direcionamento corporativo.",
            "Estruture o documento em: 1. Resumo Executivo, 2. KPIs Globais, 3. Riscos Críticos e 4. Três Ações Prioritárias.",
        ],
    ),
    "sales": dict(
        name="Commercial Sales Reporter",
        tools=[],
        instructions=[
            "Você é o Diretor Comercial de Vendas da Superstore.",
            "Analise a performance por Região e por Segmento de Clientes (Consumer, Corporate, Home Office).",
            "Apresente tabelas de vendas e formule recomendações táticas para aumentar a rentabilidade e conter descontos excessivos.",
        ],
    ),
    "ops": dict(
        name="Supply Chain & Merchandising Reporter",
        tools=[],
        instructions=[
            "Você é o Diretor de Operações e Logística.",
            "Concentre-se na rentabilidade das subcategorias críticas (especialmente itens deficitários) e na eficiência dos modais de envio.",
            "Apresente recomendações operacionais sobre estoque, controle de descontos e prazos de entrega.",
        ],
    ),
}


def build_agent(role: str, model_id: str) -> Agent:
    spec = AGENT_SPECS[role]
    return Agent(
        name=spec["name"],
        model=Gemini(id=model_id),
        tools=spec["tools"] or None,
        instructions=spec["instructions"],
        markdown=True,
    )


class ModelUnavailable(Exception):
    """Modelo inexistente/indisponível (404): não adianta retentar, só trocar de modelo."""


def run_agent_with_retry(agent: Agent, prompt: str, max_retries: int = 3) -> str:
    """Retentativas com espera progressiva para 429/503; falha rápida em 404."""
    for attempt in range(max_retries):
        try:
            response = agent.run(prompt)
            if response and response.content:
                return response.content
        except Exception as e:
            err_msg = str(e)
            if "404" in err_msg or "NOT_FOUND" in err_msg:
                raise ModelUnavailable(err_msg)
            if "503" in err_msg or "429" in err_msg:
                wait_time = (attempt + 1) * 4
                print(f"   [Aviso de fila/demanda] Aguardando {wait_time}s antes de tentar novamente...")
                time.sleep(wait_time)
            else:
                raise
    raise RuntimeError("Limite de retentativas atingido para este modelo.")


# Índice do modelo em uso: se um modelo falha, as próximas etapas já partem do seguinte
_modelo_idx = 0


def run_step(role: str, prompt: str) -> str:
    """Executa uma etapa com fallback de modelo. Etapas já concluídas não são refeitas."""
    global _modelo_idx
    ultimo_erro = None
    while _modelo_idx < len(MODEL_IDS):
        model_id = MODEL_IDS[_modelo_idx]
        try:
            agent = build_agent(role, model_id)
            resultado = run_agent_with_retry(agent, prompt)
            print(f"   ✓ concluído com '{model_id}'")
            return resultado
        except Exception as e:
            ultimo_erro = e
            print(f"   ✗ Falha em '{model_id}': {str(e)[:200]}")
            _modelo_idx += 1
    raise SystemError(f"Nenhum modelo da lista conseguiu concluir a etapa. Último erro: {ultimo_erro}")


# ORQUESTRAÇÃO
relatorios = {}

print("\n -> [1/4] Coletando dossiê de dados (Analyst Agent)...")
briefing_query = (
    "Execute uma análise abrangente da base Superstore usando as ferramentas:\n"
    "1. Calcule os KPIs macro executivos.\n"
    "2. Extraia o desempenho agrupado por 'Region', 'Segment' e 'Category'.\n"
    "3. Extraia o diagnóstico de subcategorias críticas e modais de frete.\n"
    "Traga todos os dados tabulados e consolidados."
)
dossie_dados = run_step("analyst", briefing_query)
time.sleep(2)

print(" -> [2/4] Redigindo Relatório do CEO...")
relatorios["relatorio_ceo.md"] = run_step(
    "ceo",
    f"Utilize rigorosamente os dados consolidados abaixo para redigir o Relatório Executivo do CEO:\n\n{dossie_dados}",
)
time.sleep(2)

print(" -> [3/4] Redigindo Relatório de Vendas...")
relatorios["relatorio_vendas.md"] = run_step(
    "sales",
    f"Utilize os dados consolidados abaixo para redigir o Relatório Tático de Vendas:\n\n{dossie_dados}",
)
time.sleep(2)

print(" -> [4/4] Redigindo Relatório de Produtos e Logística...")
relatorios["relatorio_produtos_logistica.md"] = run_step(
    "ops",
    f"Utilize os dados consolidados abaixo para redigir o Relatório Operacional de Produtos e Logística:\n\n{dossie_dados}",
)

print("✓ Todos os agentes executados com êxito!")

# 6. GRAVAÇÃO DOS RESULTADOS
print("\n Salvando relatórios em formato Markdown na pasta 'outputs/'...")
relatorios["dossie_dados.md"] = dossie_dados  # útil para auditoria dos números
for nome_arquivo, conteudo in relatorios.items():
    caminho = os.path.join("outputs", nome_arquivo)
    with open(caminho, "w", encoding="utf-8") as f:
        f.write(conteudo)
    print(f"✓ Arquivo gerado: {caminho}")

print("\n" + "=" * 70)
print(" PIPELINE MULTIAGENTE CONCLUÍDO COM SUCESSO!")
print("=" * 70)

# Exibição do relatório do CEO
display(Markdown("### Relatório do CEO Gerado"))
display(Markdown(relatorios["relatorio_ceo.md"]))

1/5 Instalando bibliotecas necessárias...

2/5 Configurando chaves de acesso e diretórios...
Insira sua Google API Key: ··········


✓ Chaves de autenticação configuradas.

3/5 Carregando o arquivo de dados...
✓ Arquivo localizado: SampleSuperstore.csv
✓ Base carregada com sucesso: 9994 linhas e 14 colunas.

4/5 Preparando ferramentas analíticas para os agentes...

5/5 Configurando e executando os agentes...

 -> [1/4] Coletando dossiê de dados (Analyst Agent)...
   ✓ concluído com 'gemini-3.5-flash-lite'


 -> [2/4] Redigindo Relatório do CEO...
   ✓ concluído com 'gemini-3.5-flash-lite'


 -> [3/4] Redigindo Relatório de Vendas...
   ✓ concluído com 'gemini-3.5-flash-lite'


 -> [4/4] Redigindo Relatório de Produtos e Logística...
   ✓ concluído com 'gemini-3.5-flash-lite'
✓ Todos os agentes executados com êxito!

 Salvando relatórios em formato Markdown na pasta 'outputs/'...
✓ Arquivo gerado: outputs/relatorio_ceo.md
✓ Arquivo gerado: outputs/relatorio_vendas.md
✓ Arquivo gerado: outputs/relatorio_produtos_logistica.md
✓ Arquivo gerado: outputs/dossie_dados.md

 PIPELINE MULTIAGENTE CONCLUÍDO COM SUCESSO!


### Relatório do CEO Gerado

# Relatório Executivo Estratégico

**Para:** CEO e Acionistas  
**De:** Chief of Staff  
**Data:** Atualização Consolidada Superstore  

---

### 1. Resumo Executivo
A operação global da Superstore encerrou o ciclo com um **Faturamento Total de USD 2,297,200.86** e um **Lucro Total de USD 286,397.02**, resultando em uma **Margem Líquida Geral de 12.47%** sobre 9,994 pedidos. Embora os indicadores macro mostrem saúde financeira aparente, o diagnóstico dimensional revela assimetrias severas: a categoria de *Furniture* (especialmente *Tables*) e a região *Central* operam como drenos críticos de margem, mascarados pelo forte desempenho de *Technology* e pela região *West*. A estratégia imediata exige o estancamento de perdas operacionais e a revisão rigorosa da política de descontos.

---

### 2. KPIs Globais
* **Faturamento Total:** $2,297,200.86
* **Lucro Total:** $286,397.02
* **Margem Líquida Geral:** 12.47%
* **Desconto Médio Concedido:** 15.62%
* **Volume de Pedidos Únicos:** 9,994

---

### 3. Riscos Críticos
* **Destruição de Margem em Furniture:** A categoria responde por 32.30% das vendas ($742.000), mas entrega apenas 2.49% de margem ($18.451,30), puxada para o vermelho pelas subcategorias *Tables* (-8.56% de margem) e *Bookcases* (-3.02%).
* **Erosão por Descontos Excessivos:** O desconto médio de 15.62% atinge patamares destrutivos em itens de baixo retorno, como *Tables* (26.13% de desconto) e *Machines* (30.61% de desconto), inviabilizando o lucro operacional.
* **Ineficiência Geográfica (Região Central):** A região *Central* apresenta a menor margem líquida da companhia (7.92%), gerando apenas $39.706,40 de lucro em cima de $501.240 em vendas, refletindo falhas estruturais de precificação e dinâmicas locais desfavoráveis.

---

### 4. Três Ações Prioritárias
1. **Revisão e Teto de Descontos (*Discount Cap*):** Implementar travas automáticas no ERP para limitar descontos nas subcategorias críticas (*Tables*, *Bookcases* e *Machines*), alinhando a política comercial aos limiares de rentabilidade mínima.
2. **Descontinuação ou Reestruturação de *Tables*:** Suspender imediatamente as promoções na subcategoria *Tables* e reavaliar custos logísticos e de fornecedores; caso a reversão do prejuízo seja inviável, descontinuar a linha.
3. **Turnaround da Região *Central*:** Auditar a operação logística, mix de produtos e política comercial da região *Central* para elevar sua margem ao patamar corporativo (>12%), ou realocar capital de marketing para as regiões de alta performance (*West* e *East*).